# Actor Clustering - GAE + KMeans

This notebook implements the actor-level clustering pipeline for one model using the shared preprocessing and graph construction logic.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.clustering.gae import run_gae_kmeans
from src.preprocessing.pipeline import prepare_common_actor_data
from src.utils.config import RESULTS_DIR, SEED
from src.utils.io_utils import save_json, save_results_csv

MAX_NODES = 2000
N_CLUSTERS = 8
common = prepare_common_actor_data(max_nodes=MAX_NODES)
assignments, embedding, metrics = run_gae_kmeans(
    common["graph"], common["feature_matrix"], common["node_order"], n_clusters=N_CLUSTERS, seed=SEED
)
result_dir = RESULTS_DIR / "gae_kmeans"
save_results_csv(assignments, result_dir / "cluster_assignments.csv")
save_json(metrics, result_dir / "metrics.json")
import numpy as np
np.save(result_dir / "embeddings.npy", embedding)
print(metrics)

/home/vinhnguyen/Workspace/Personal/elliptic-actor-graph/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:484: UserWarning: Found GPU0 NVIDIA GeForce 840M which is of compute capability (CC) 5.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Your installed torch==2.14.0+cu130 does not include kernels for this GPU. Reinstall the same version against a CUDA build that does, e.g.:
  For CUDA 12.6 use pip install torch==2.14.0 --index-url https://download.pytorch.org/whl/cu126
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/vinhnguyen/Workspace/Personal/elliptic-actor-graph/.venv/lib/python3.13/site-packages/to

{'method': 'gae_kmeans', 'n_nodes': 2000, 'n_edges': 1031, 'n_clusters': 8, 'embedding_dimensions': 32, 'epochs': 40, 'runtime_seconds': 17.534639871002582, 'seed': 42, 'final_loss': 0.6932970285415649, 'loss_history': [1.306019902229309, 0.9891998767852783, 0.8535763621330261, 0.7963966727256775, 0.7699555158615112, 0.7538285255432129, 0.7417649030685425, 0.7338190674781799, 0.726752519607544, 0.7205655574798584, 0.7157020568847656, 0.7119520902633667, 0.7088733315467834, 0.7063103914260864, 0.7043617963790894, 0.7028987407684326, 0.7016246318817139, 0.7005054354667664, 0.6994318962097168, 0.6983720660209656, 0.6974032521247864, 0.6966204047203064, 0.6959611177444458, 0.6954171657562256, 0.6949506998062134, 0.6945586204528809, 0.6942570209503174, 0.6940036416053772, 0.6938506364822388, 0.6937428712844849, 0.6936589479446411, 0.6935884952545166, 0.6935253739356995, 0.6934727430343628, 0.6934285163879395, 0.6933943629264832, 0.6933656334877014, 0.6933398246765137, 0.6933178901672363, 0.

In [2]:
from src.clustering.evaluation import evaluate_clusters
from src.clustering.visualization import save_cluster_figures

# Re-run with an explicit checkpoint so the trained encoder is persisted.
assignments, embedding, metrics = run_gae_kmeans(
    common["graph"], common["feature_matrix"], common["node_order"],
    n_clusters=N_CLUSTERS, seed=SEED,
    checkpoint_path=str(result_dir / "model.pt"),
)
labels = common["actor_df"].set_index("actor_id")["class"].reindex(assignments["actor_id"])
metrics.update(evaluate_clusters(assignments, embedding, labels))
metrics["research_questions"] = [
    "Does combining actor features with AddrAddr structure improve representation quality?",
    "How does GAE compare with Node2Vec when evaluated in the same actor space?",
]
save_results_csv(assignments, result_dir / "cluster_assignments.csv")
save_json(metrics, result_dir / "metrics.json")
np.save(result_dir / "embeddings.npy", embedding)
save_cluster_figures(assignments, result_dir, embedding, metrics["loss_history"], "GAE actor communities")
print("Research interpretation metrics:", {key: metrics[key] for key in ("final_loss", "silhouette", "ari", "nmi", "ami")})

Research interpretation metrics: {'final_loss': 0.6932970285415649, 'silhouette': 0.7507330179214478, 'ari': -0.01485177111803307, 'nmi': 0.0030226235091749304, 'ami': -0.0029675924538248135}
